# Same questions, different memory: history
Review evidence, not a fresh experiment. Recomputes point estimates from curated numeric exports; does not rerun embeddings, Jev, timing or bootstrap intervals. Source hashes are in the eval file. Source data and notebooks for independent end-to-end reproduction remain a release prerequisite.


In [ ]:
import json
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
data = json.loads((root / 'docs/media/same-memory/history-eval.json').read_text())


In [ ]:
def auc(rows, column, refund=False):
    numerator = denominator = 0.
    for company in {r[0] for r in rows}:
        subset = [r for r in rows if r[0] == company and (not refund or r[5])]
        ties = {}
        for r in subset:
            score = data['label_rates'][r[1]] if column == 'M' else r[2]
            counts = ties.setdefault(score, [0, 0])
            counts[0] += r[3]
            counts[1] += r[4]
        negatives = positives = wins = 0.
        for score, (n, p) in sorted(ties.items()):
            wins += p * (negatives + n / 2)
            negatives += n
            positives += p
        numerator += wins
        denominator += negatives * positives
    return numerator / denominator
for population, refund in [('primary', False), ('picture', True)]:
    for arm in ['M', 'V']:
        value = auc(data['histogram'], arm, refund)
        assert abs(value - data['metrics'][population][arm]) < 1e-10
        print(population, arm, f'{value:.3f}')
assert data['gate']['passed']
print('Published example rows:', data['published']['rows'], data['published']['source'])
print('Archive wipe remains an illustration, not a measured service result.')
